STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


STEP 2: Set up paths and install dependencies
/content/drive/MyDrive/PCOS_dataset/
   ├── Normal/
   ├── PCOS/


In [ ]:
!pip install tensorflow numpy scikit-learn pyswarms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pyswarms as ps

DATA_DIR = "/content/drive/MyDrive/PCOS MINOR/data/train"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 128
NUM_CLIENTS = 10
LOCAL_EPOCHS = 2
COMMS_ROUNDS = 20
LEARNING_RATE = 1e-4
OUTPUT_WEIGHTS_FILE = "/content/drive/MyDrive/PCOS MINOR/Federated Learning using ResNet50 (20 CR, 0.0001 LR, 128 BS, PSO).h5"

STEP 3: Helper functions

In [ ]:
def build_resnet50_model(num_classes, input_shape=(224,224,3), dropout_rate=0.3):
    base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=input_shape, pooling='avg')
    base.trainable = False
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.applications.resnet.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    return model

def gather_filepaths_and_labels(data_dir):
    classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    paths, labels = [], []
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for f in os.listdir(cls_dir):
            fpath = os.path.join(cls_dir, f)
            if os.path.isfile(fpath):
                paths.append(fpath)
                labels.append(idx)
    return np.array(paths), np.array(labels), classes

def paths_to_dataset(paths, labels, batch_size=16, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths))
    def load_img(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, IMAGE_SIZE)
        img = tf.keras.applications.resnet.preprocess_input(img)
        return img, label
    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def shard_data(paths, labels, num_clients):
    idx = np.arange(len(paths))
    np.random.shuffle(idx)
    paths, labels = paths[idx], labels[idx]
    shards = []
    n = len(paths) // num_clients
    for i in range(num_clients):
        start, end = i * n, (i + 1) * n if i != num_clients - 1 else len(paths)
        shards.append((paths[start:end], labels[start:end]))
    return shards

def scale_weights(weights, scalar):
    return [w * scalar for w in weights]

def aggregate_scaled_weights(scaled_weights_list):
    avg = []
    for layer in zip(*scaled_weights_list):
        avg.append(np.sum(layer, axis=0))
    return avg


STEP 4: Define PSO optimizer for aggregation weights

In [ ]:
def pso_optimize_weights(local_accuracies):
    # local_accuracies = [acc1, acc2, acc3, ...]
    num_clients = len(local_accuracies)

    def objective_function(x):
        # each row of x = one particle's weights
        # weights should sum to 1
        fitness = []
        for weights in x:
            weights = np.clip(weights, 0, 1)
            weights = weights / np.sum(weights)
            score = -np.dot(weights, local_accuracies)  # we minimize negative accuracy
            fitness.append(score)
        return np.array(fitness)

    # PSO configuration
    options = {'c1': 0.8, 'c2': 0.9, 'w': 0.5}
    optimizer = ps.single.GlobalBestPSO(n_particles=20, dimensions=num_clients, options=options)
    best_cost, best_pos = optimizer.optimize(objective_function, iters=50, verbose=False)
    best_pos = np.clip(best_pos, 0, 1)
    best_pos = best_pos / np.sum(best_pos)
    return best_pos


STEP 5: PSO-based Federated Training

In [ ]:
def run_fedavg_pso():
    # Load dataset
    paths, labels, classes = gather_filepaths_and_labels(DATA_DIR)
    print(f"Found {len(classes)} classes: {classes}")

    # Split into train/test
    p_train, p_test, y_train, y_test = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=42)
    shards = shard_data(p_train, y_train, NUM_CLIENTS)
    clients = {f"client_{i+1}": {"paths": shards[i][0], "labels": shards[i][1]} for i in range(NUM_CLIENTS)}

    # Build global model
    global_model = build_resnet50_model(num_classes=len(classes))
    global_model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                         loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                         metrics=['accuracy'])

    test_ds = paths_to_dataset(p_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

    for rnd in range(1, COMMS_ROUNDS + 1):
        print(f"\n--- Communication Round {rnd}/{COMMS_ROUNDS} ---")
        local_weights, local_accuracies = [], []

        # Train each client locally
        for cname, data in clients.items():
            local_ds = paths_to_dataset(data["paths"], data["labels"], batch_size=BATCH_SIZE)
            local_model = tf.keras.models.clone_model(global_model)
            local_model.build((None, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
            local_model.set_weights(global_model.get_weights())
            local_model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                                metrics=['accuracy'])
            history = local_model.fit(local_ds, epochs=LOCAL_EPOCHS, verbose=0)
            acc = np.mean(history.history['accuracy'])
            local_accuracies.append(acc)
            local_weights.append(local_model.get_weights())
            tf.keras.backend.clear_session()

        # Optimize aggregation weights via PSO
        best_weights = pso_optimize_weights(local_accuracies)

        # Scale local weights using PSO-optimized weights
        scaled_local_weights = []
        for w, scale in zip(local_weights, best_weights):
            scaled_local_weights.append(scale_weights(w, scale))

        # Aggregate to update global model
        new_weights = aggregate_scaled_weights(scaled_local_weights)
        global_model.set_weights(new_weights)

        # Evaluate global model
        loss, acc = global_model.evaluate(test_ds, verbose=0)
        print(f"Round {rnd} - Global Test Loss: {loss:.4f} | Accuracy: {acc:.4%}")
        print(f"PSO Aggregation Weights: {np.round(best_weights, 3)}")

    global_model.save(OUTPUT_WEIGHTS_FILE)
    print(f"\n✅ PSO-FedAvg complete! Model saved to {OUTPUT_WEIGHTS_FILE}")

run_fedavg_pso()

Found 2 classes: ['infected', 'notinfected']
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- Communication Round 1/20 ---
Round 1 - Global Test Loss: 0.7721 | Accuracy: 56.1039%
PSO Aggregation Weights: [0.454 0.    0.    0.    0.    0.    0.056 0.037 0.    0.454]

--- Communication Round 2/20 ---
Round 2 - Global Test Loss: 0.5369 | Accuracy: 76.8831%
PSO Aggregation Weights: [0.24  0.145 0.    0.    0.    0.147 0.24  0.    0.    0.229]

--- Communication Round 3/20 ---
Round 3 - Global Test Loss: 0.4131 | Accuracy: 85.9740%
PSO Aggregation Weights: [0.306 0.205 0.    0.    0.    0.2   0.29  0.    0.    0.   ]

--- Communication Round 4/20 ---
Round 4 - Global Test Loss: 0.3512 | Accuracy: 87.2727%
PSO Aggregation Weights: [0.272 0.322 0.    0.    0.289 0.    0.    0.    0.117 0.   ]

--- Communication Round 5/20 ---
Round 5 - Global Test Loss: 0.3112 | Accuracy: 89.3506%
PSO Aggregation Weights: [0.543 0.309 0.    0.    0.148 0.    0.    0.    0.    0.   ]

--- Communication 

2025-11-15 07:13:36,864 - absl - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


Round 20 - Global Test Loss: 0.0646 | Accuracy: 99.7403%
PSO Aggregation Weights: [0.203 0.32  0.    0.32  0.039 0.    0.    0.    0.    0.118]

✅ PSO-FedAvg complete! Model saved to /content/drive/MyDrive/PCOS MINOR/Federated Learning using ResNet50 (20 CR, 0.0001 LR, 128 BS, PSO).h5
